In [1]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

print("Model ready!")

Model ready!


In [2]:
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(
    max_results=3,
    search_depth="advanced",
    include_answer=True,
)

print("Tavily search tool ready!")

Tavily search tool ready!


In [3]:
# Basic Middleware Example: Request/Response Logger
from langchain.agents.middleware import wrap_model_call
from datetime import datetime

@wrap_model_call
def logging_middleware(request, handler):
    """Simple middleware that logs all model interactions."""
    print(f" [{datetime.now().strftime('%H:%M:%S')}] Model Request:")
    print(f"   Model: {request.model}")
    print(f"   Messages: {len(request.state.get('messages', []))} messages")
    
    # Call the actual model (this is where the magic happens)
    response = handler(request)
    
    print(f"✅ [{datetime.now().strftime('%H:%M:%S')}] Model Response received")
    print(f"   Response type: {type(response)}")
    print("-" * 50)
    
    return response

# Create agent with logging middleware
basic_agent_with_logging = create_agent(
    model=model,
    tools=[tavily_tool],
    middleware=[logging_middleware],  # Apply our middleware
    system_prompt=(
        "You are a helpful research assistant with web search capabilities. "
        "Use Tavily search when the user needs current or factual information from the web."
    ),
)

In [4]:
# Test the logging middleware in action
print("=== Testing Logging Middleware ===")

result = basic_agent_with_logging.invoke({
    "messages": [{"role": "user", "content": "Search the web: what are the latest headlines in AI this week?"}]
})

print("\nFinal Result:")
print(result["messages"][-1].content)

=== Testing Logging Middleware ===
 [11:42:23] Model Request:
   Model: metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16', 'langchain-openai': '1.6.0'}} profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} client=<openai.resources.chat.completions.completions.Completions object at 0x000001F902402CF0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001F9024037

In [5]:
# Dynamic model example - Now you understand middleware!
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

# Set up different models for different scenarios
basic_model = ChatOpenAI(model="gpt-4.1-mini")
advanced_model = ChatOpenAI(model="gpt-4.1")

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose model based on conversation complexity."""
    message_count = len(request.state["messages"])
    
    # Check if the question involves complex calculations or multiple steps
    last_message = request.state["messages"][-1].content if request.state["messages"] else ""
    complex_keywords = ["complex", "multiple", "detailed", "analysis", "explain"]
    
    if message_count > 5 or any(keyword in last_message.lower() for keyword in complex_keywords):
        # Use advanced model for complex scenarios
        print("Using advanced model (GPT-4.1) for complex task")
        request.model = advanced_model
    else:
        # Use basic model for simple tasks
        print("Using basic model (GPT-4.1-mini) for simple task")
        request.model = basic_model
    
    return handler(request)

# Create dynamic agent
dynamic_agent = create_agent(
    model=basic_model,  # Default model
    tools=[tavily_tool],
    middleware=[dynamic_model_selection],  # This is middleware in action!
    system_prompt=(
        "You are an intelligent research assistant that adapts to task complexity. "
        "Use Tavily search for questions that need up-to-date information from the web."
    ),
)

print("Dynamic agent created successfully!")

Dynamic agent created successfully!


In [6]:
# Test dynamic agent with simple task
print("Testing with simple task:")
result = dynamic_agent.invoke({
    "messages": [{"role": "user", "content": "What is LangChain?"}]
})
print(result["messages"][-1].content)
print("\n" + "="*50 + "\n")

# Test dynamic agent with complex task
print("Testing with complex task:")
result = dynamic_agent.invoke({
    "messages": [{"role": "user", "content": "Please provide a detailed analysis comparing recent AI agent frameworks and their middleware support."}]
})
print(result["messages"][-1].content)

Testing with simple task:
Using basic model (GPT-4.1-mini) for simple task


C:\Users\localadmin\AppData\Local\Temp\ipykernel_6292\1151648309.py:24: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = basic_model


LangChain is a framework designed to simplify the creation of applications using large language models (LLMs) by enabling developers to build language model-driven applications with ease. It provides tools and components that facilitate the integration of language models into various workflows and applications. LangChain supports chaining together different components such as LLMs, prompts, memory, and external data sources to create complex applications like chatbots, question-answering systems, and more. It helps manage the complexities of working with language models, such as prompt management, memory handling, and connecting to external APIs or data stores.


Testing with complex task:
Using advanced model (GPT-4.1) for complex task


C:\Users\localadmin\AppData\Local\Temp\ipykernel_6292\1151648309.py:20: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = advanced_model


Using advanced model (GPT-4.1) for complex task


C:\Users\localadmin\AppData\Local\Temp\ipykernel_6292\1151648309.py:20: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = advanced_model


Here’s a detailed analysis comparing recent AI agent frameworks and their middleware support, synthesizing insights from leading technical sources:

1. Key Frameworks Overview

- LangGraph (built atop DeepAgents): Offers advanced, stateful workflows supporting multi-turn, memory-intensive conversations.
- CrewAI: Excels in multi-agent orchestration with central coordination and limited but high-performance middleware.
- Microsoft Semantic Kernel: Lightweight SDK with strong middleware capabilities for enterprise integration and plugin chaining.
- LlamaIndex, Microsoft AutoGen, OpenAI Swarm: Feature varying levels of native middleware and multi-agent support for specific scenarios (data indexing, multi-agent conversations, lightweight experimentation, respectively).

2. Middleware Support: Definitions and Differences

- Middleware in AI agent frameworks refers to a layer/tools used for transforming, managing, or intercepting the data and control flow between AI agents and the backend or

In [7]:
import time
from langchain.agents.middleware import wrap_tool_call
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool

_fetch_attempts = {"count": 0}

@tool
def fetch_live_external_data(topic: str) -> str:
    """Fetch live or real-time external data for a topic (e.g. weather, stock quote, breaking news).

    Use this when the user asks for up-to-the-minute or live information.
    """
    _fetch_attempts["count"] += 1
    attempt = _fetch_attempts["count"]
    print(f"fetch_live_external_data call #{attempt} for topic={topic!r}")

    # Demo: fail the first two times, succeed on the third so retry middleware can recover
    if attempt <= 2:
        raise ConnectionError("Simulated failure: external data service is unreachable.")

    return (
        f"Live data for '{topic}': Bengaluru (Bangalore) is currently 27°C, "
        "partly cloudy, with a light breeze from the east."
    )

@wrap_tool_call
def handle_tool_errors(request, handler):
    """Retry a failed tool call up to 3 times; if all attempts fail, return a user-facing ToolMessage."""
    max_attempts = 3
    retry_delay_seconds = 2
    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            print(f"Tool call attempt {attempt}/{max_attempts}")
            return handler(request)
        except Exception as e:
            last_error = e
            print(f"Attempt {attempt} failed: {e}")
            if attempt < max_attempts:
                print(f"Retrying in {retry_delay_seconds} seconds...")
                time.sleep(retry_delay_seconds)

    msg = str(last_error).lower()
    if "connection" in msg or "timeout" in msg or "unreachable" in msg:
        return ToolMessage(
            content=(
                "The live data service is temporarily unavailable after 3 attempts. "
                "Answer from general knowledge or suggest the user try again later."
            ),
            tool_call_id=request.tool_call["id"],
        )
    return ToolMessage(
        content=f"That tool hit an error after 3 attempts: {last_error!s}. Please rephrase or use another approach.",
        tool_call_id=request.tool_call["id"],
    )

print("Demo flaky tool and retry middleware ready.")

Demo flaky tool and retry middleware ready.


In [8]:
# Agent: Tavily search works; live-data tool fails twice then succeeds (retry middleware)
error_handling_agent = create_agent(
    model=model,
    tools=[fetch_live_external_data, tavily_tool],
    middleware=[handle_tool_errors],
    system_prompt=(
        "You are a helpful assistant. Use fetch_live_external_data when the user wants live or real-time external data. "
        "Use Tavily search for general web research. If a tool result indicates a service error, say so clearly and offer alternatives."
    ),
)

print("Testing retries (tool fails twice, succeeds on the third attempt):")
result = error_handling_agent.invoke({
    "messages": [{"role": "user", "content": "What is the exact temperature in Bangalore, India right now?"}]
})

print(result["messages"][-1].content)
print("\n" + "=" * 50 + "\n")

Testing retries (tool fails twice, succeeds on the third attempt):
Tool call attempt 1/3
fetch_live_external_data call #1 for topic='current temperature in Bangalore, India'
Attempt 1 failed: Simulated failure: external data service is unreachable.
Retrying in 2 seconds...
Tool call attempt 2/3
fetch_live_external_data call #2 for topic='current temperature in Bangalore, India'
Attempt 2 failed: Simulated failure: external data service is unreachable.
Retrying in 2 seconds...
Tool call attempt 3/3
fetch_live_external_data call #3 for topic='current temperature in Bangalore, India'
The current temperature in Bangalore, India is 27°C.




In [9]:
# Test normal operation - Tavily search works fine
print("Testing normal operation (Tavily search):")
result = error_handling_agent.invoke({
    "messages": [{"role": "user", "content": "Search the web: who won the latest Formula 1 race?"}]
})
print(result["messages"][-1].content)

Testing normal operation (Tavily search):
Tool call attempt 1/3
The latest Formula 1 race winner is Lewis Hamilton, who secured his first win for Ferrari at the Barcelona-Catalunya Grand Prix on June 14, 2026. This victory ended Kimi Antonelli's five-race winning streak. Hamilton, at 41, became the oldest Formula 1 race winner since 1970.


In [10]:
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.chat_models import init_chat_model

In [11]:
@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """Send an email to a recipient. This is a sensitive operation that requires approval."""
    return f"Email sent to {recipient} with subject '{subject}' and body: {body[:50]}..."

@tool
def search_database(query: str) -> str:
    """Search the company database for information."""
    return f"Database search results for '{query}': Found 5 matching records"

@tool
def delete_record(record_id: str) -> str:
    """Delete a record from the database. This is a sensitive operation."""
    return f"Record {record_id} has been deleted from the database"

print("Guardrail tools created successfully!")

Guardrail tools created successfully!


In [12]:
# Create an agent with PII detection guardrails
pii_agent = create_agent(
    model=init_chat_model("gpt-4.1-mini"),
    tools=[search_database],
    middleware=[
        # Mask credit cards in user input before sending to model
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Mask API keys in user input
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",  # OpenAI API key pattern
            strategy="mask",
            apply_to_input=True,
        ),
    ],
)

print("PII-protected agent created!")

PII-protected agent created!


In [13]:
# Test PII detection with credit card and API key
print("=== Testing PII Detection ===")
print("Input: 'My card is 4111 1111 1111 1111 and my API key is sk-abcdefghijklmnopqrstuvwxyz123456'")
print()

try:
    result = pii_agent.invoke({
        "messages": [{
            "role": "user",
            "content": "My card is 4111 1111 1111 1111 and my API key is sk-abcdefghijklmnopqrstuvwxyz123456. Can you search the database for my recent orders?"
        }]
    })

    print("Agent Response:")
    print(result['messages'])

except Exception as e:
    print(f"Error: {e}")

=== Testing PII Detection ===
Input: 'My card is 4111 1111 1111 1111 and my API key is sk-abcdefghijklmnopqrstuvwxyz123456'

Agent Response:
[HumanMessage(content='My card is **** **** **** 1111 and my API key is ****3456. Can you search the database for my recent orders?', additional_kwargs={}, response_metadata={}, id='ad0a3655-c4c9-4227-a521-383f86926c62'), AIMessage(content='For your security and privacy, I cannot process or store sensitive information such as your full card number or API key. However, if you would like, I can help you search the database for your recent orders. Please provide me with your user ID, email address, or any other non-sensitive identifier associated with your account.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 72, 'total_tokens': 137, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_token

In [14]:
# Create an agent with human-in-the-loop for sensitive operations
hitl_agent = create_agent(
    model=init_chat_model("gpt-4o-mini"),
    tools=[send_email, search_database, delete_record],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # Require approval for sensitive operations
                "send_email": True,
                "delete_record": True,
                # Auto-approve safe operations
                "search_database": False,
            }
        ),
    ],
    # Persist the state across interrupts
    checkpointer=InMemorySaver(),
)

print("Human-in-the-loop agent created!")

Human-in-the-loop agent created!


In [15]:
# Test human-in-the-loop with a safe operation (should execute immediately)
print("=== Testing Safe Operation (No Approval Needed) ===")
print("Request: Search the database for quarterly sales reports")
print()

config = {"configurable": {"thread_id": "safe_thread"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Search the database for quarterly sales reports"}]},
    config=config
)

print("Agent Response:")
result['messages']

=== Testing Safe Operation (No Approval Needed) ===
Request: Search the database for quarterly sales reports

Agent Response:


[HumanMessage(content='Search the database for quarterly sales reports', additional_kwargs={}, response_metadata={}, id='c4c04688-282f-40e8-9b68-76ecc3c54909'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 117, 'total_tokens': 134, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c73dd82b09', 'id': 'chatcmpl-EHNlvDC8dDpGFzuvXNj1sDS6jLMhV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a041e1-13c4-76c3-a2cb-4c166e62c870-0', tool_calls=[{'name': 'search_database', 'args': {'query': 'quarterly sales reports'}, 'id': '

In [16]:
# Test human-in-the-loop with a sensitive operation (requires approval)
print("=== Testing Sensitive Operation (Requires Approval) ===")
print("Request: Send an email to the team")
print()

config = {"configurable": {"thread_id": "sensitive_thread"}}

# This will pause and wait for approval
result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the quarterly meeting"}]},
    config=config
)

print("Initial Response (Waiting for Approval):")
print(f"Messages: {len(result.get('messages', []))}")
print(f"Next step: {result.get('next', 'Unknown')}")
print("\n⏸️ Agent paused, waiting for human approval...")

=== Testing Sensitive Operation (Requires Approval) ===
Request: Send an email to the team

Initial Response (Waiting for Approval):
Messages: 2
Next step: Unknown

⏸️ Agent paused, waiting for human approval...


In [ ]:
# Approve the sensitive operation
print("=== Human Approval: APPROVE ===")
print()

# Resume with approval
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config  # Same thread ID to resume
)

print("Final Response After Approval:")
approved_result['messages']

In [21]:
# Demonstrate rejection
print("=== Testing Rejection ===")
print("Request: Delete a record")
print()

config_reject = {"configurable": {"thread_id": "reject_thread1"}}

# Start a sensitive operation
result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete record ID-12345 from the database"}]},
    config=config_reject
)

print("⏸️ Agent paused for approval...")
print("Human Decision: REJECT")
print()

# Reject the operation
rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject"}]}),
    config=config_reject
) 
print("Final Response After Rejection:")
rejected_result['messages']

=== Testing Rejection ===
Request: Delete a record

⏸️ Agent paused for approval...
Human Decision: REJECT

Final Response After Rejection:


[HumanMessage(content='Delete record ID-12345 from the database', additional_kwargs={}, response_metadata={}, id='b7f0c770-bfb2-4ea7-a220-96f87b65da75'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 119, 'total_tokens': 137, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c73dd82b09', 'id': 'chatcmpl-EHNnR2cLdZ8YN7Ly0jILbBKJrUyOC', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a041e2-818c-7001-a979-051d4583f481-0', tool_calls=[{'name': 'delete_record', 'args': {'record_id': 'ID-12345'}, 'id': 'call_98NemyEyzxjyEVz